# Stochastic Variational Consistency Model (StochVarCM)
## Estimation de $\sigma(t;\mathbf{x})$ — Variance du processus de débruitage

Extension stochastique de **VarCM** (`Notebook_consistency_model_VarCM_spde`).

**Apport principal :** Le UNet prédit l'**exposant $\alpha(\mathbf{x})$** d'une loi de puissance qui
paramétrise la variance du mapping $\mathbf{x}_t \to \mathbf{x}_0$ en fonction du temps solveur $t$.

---

## 1. Rappel : Pairwise Consistency & Dynamique Variationnelle (VarCM)

### 1.1 Opérateur de transport pairwise

$$
g_\phi(\mathbf{x}_t,\, \mathbf{c},\, t, t') =
c_\text{skip}(t,t')\,\mathbf{x}_t + c_\text{out}(t,t')\,F_\theta\!\left(c_\text{in}(t)\,\mathbf{x}_t,\;\mathbf{c},\;t,\;t'\right)
$$

Préconditionnement EDM avec interpolant linéaire ($\alpha(t)=t$, $\beta(t)=1-t$) :

$$
c_\text{in}(t) = \frac{1}{\sqrt{\sigma_\text{data}^2(1-t)^2 + \sigma_\text{noise}^2\,t^2}},\quad
c_\text{skip}(t,t') = \frac{t'}{t},\quad
c_\text{out}(t,t') = (1-t') - c_\text{skip}\,(1-t)
$$

Convention de temps : $t=1$ = bruit pur, $t=0$ = état propre.

### 1.2 Dynamique variationnelle

$$
J(\mathbf{x}) = J_o(\mathbf{x},\mathbf{y}) + \lambda\,J_b(\mathbf{x}),
\qquad \frac{d\mathbf{x}_t}{dt} = -\nabla_\mathbf{x} J(\mathbf{x}_t)
$$

Le conditionnement $\mathbf{c}$ est construit à partir de $-\nabla J$ ou des observations $\mathbf{y}$ (mode `obs`).

---

## 2. Extension Stochastique

### 2.1 Motivation : incertitude croissante vers $t=1$

While the deterministic formulation leads to a contractive dynamics converging toward the MAP estimator, it does not capture posterior uncertainty. In variational data assimilation, the posterior distribution around the optimal trajectory may exhibit significant dispersion, which must be explicitly modeled. We therefore extend the pairwise consistency framework to a stochastic setting.

La difficulté du mapping $\mathbf{x}_t \to \mathbf{x}_0$ dépend fortement de $t$ :

| Temps solveur $t$ | État de $\mathbf{x}_t$ | Incertitude du débruitage |
|---|---|---|
| $t \approx 1$ | Bruit pur | **Maximale** — infiniment de $\mathbf{x}_0$ compatibles |
| $t \approx 0.5$ | Mi-chemin | Intermédiaire |
| $t \approx 0$ | Quasi-propre | **Minimale** — quasi-déterministe |

### 2.2 Opérateur de transport stochastique

We retain the same pairwise operator structure, but extend the network to predict both a mean field and a scalar uncertainty exponent:

$$
F_\theta\!\left(c_\text{in}(t)\,\mathbf{x}_t,\;\mathbf{c},\;t,\;t'\right)
=
\big(\boldsymbol{\mu}_\phi(\mathbf{x}_t,t,t'),\;\alpha_\text{raw}(\mathbf{x}_t)\big)
$$

The deterministic transport remains identical to VarCM:

$$
g_\phi^\text{det}(\mathbf{x}_t, t, t')
=
c_\text{skip}(t,t')\,\mathbf{x}_t + c_\text{out}(t,t')\,\boldsymbol{\mu}_\phi(\mathbf{x}_t,t,t')
$$

The stochastic extension adds isotropic noise at each solver step via a reparameterization:

$$
\mathbf{x}_{t'}
=
g_\phi^\text{det}(\mathbf{x}_t, t, t')
+
\sigma_\phi(\mathbf{x}_t, t)\,\boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(0,\mathbf{I})
$$

This defines a stochastic transport operator that can be interpreted as a discretization of a reverse-time stochastic flow driven by the conditional posterior score.

### 2.3 Paramétrique loi de puissance — $\sigma(t;\mathbf{x}) = \sigma_\text{noise}\cdot t^{\,\alpha(\mathbf{x})}$

Plutôt qu'un $\log\sigma$ libre, on apprend l'exposant $\alpha$ de la loi de puissance :

$$
\boxed{\sigma_\phi(\mathbf{x}_t, t) = \sigma_\text{noise} \cdot t^{\,\alpha(\mathbf{x}_t)}},
\qquad
\alpha(\mathbf{x}) = \underbrace{\text{softplus}\!\bigl(\text{MLP}(\text{AvgPool}(h_\text{dec}))\bigr)}_{> 0} + \alpha_\text{min}
$$

**Garanties structurelles (impossibles à violer) :**
$$
\sigma_\phi(t=1;\mathbf{x}) = \sigma_\text{noise} \quad \text{et} \quad \sigma_\phi(t=0;\mathbf{x}) = 0 \quad \forall\,\alpha,\,\mathbf{x}
$$

**Interprétation de $\alpha$ :**

| Valeur | Forme du profil | Interprétation physique |
|--------|----------------|------------------------|
| $\alpha < 1$ | Concave (hyperbolique) | Forte incertitude dès $t$ faible — débruitage difficile même en fin de chaîne |
| $\alpha = 1$ | Linéaire | Croissance uniforme de l'incertitude avec $t$ |
| $\alpha > 1$ | Convexe (parabolique) | Incertitude concentrée sur les grands $t$ — fin de chaîne quasi-certaine |

Init : biais $= 0.541 \Rightarrow \alpha_\text{init} = \text{softplus}(0.541)+0.1 \approx 1.0$ (linéaire au départ).

### 2.4 Loss découplée : MSE pour $\boldsymbol{\mu}$, NLL pour $\sigma$

Rather than jointly training $\boldsymbol{\mu}$ and $\sigma$ via a single NLL objective, we decouple the two learning signals. This design is motivated by the pathological gradient scaling of pure NLL training:

$$
\frac{\partial \mathcal{L}_\text{NLL}}{\partial \boldsymbol{\mu}} = -\frac{\mathbf{x}_\star - \boldsymbol{\mu}}{\sigma^2(t)}
$$

With $\sigma(t{=}0.5) \approx 5$ ($\sigma_\text{noise}{=}10$), this gradient is **25× weaker** than MSE.
The network can reduce the NLL by inflating $\sigma$ (entropy term $\frac{d}{2}\log\sigma^2$) rather than improving the mean prediction — physical structures are not learned.

**Solution: stop-gradient decoupling.** The total loss combines deterministic pairwise consistency (identical to VarCM) with a single NLL term that calibrates $\sigma$ only:

$$
\mathcal{L}_\text{total}
=
\underbrace{\lambda_\text{pair}\,L_\text{pair} + \lambda_\text{anc}(L_\text{long}+L_\text{short}) + \lambda_\text{ae}\,L_\text{ae}}_{\text{MSE — identique VarCM, gradient plein sur }\boldsymbol{\mu}}
\;+\;
\underbrace{\lambda_\sigma\,L_\sigma}_{\text{NLL — calibre }\sigma\text{ seulement}}
$$

The MSE terms enforce self-consistency of the deterministic transport (pairwise jumps, anchoring to $\mathbf{x}_0$), providing strong gradients on $\boldsymbol{\mu}$. The NLL term $L_\sigma$ calibrates the predicted variance to match the actual reconstruction error, but receives **no gradient through $\boldsymbol{\mu}$** (stop-gradient):

$$
L_\sigma
=
\mathcal{L}_\text{NLL}\!\big(\mathbf{x}_0,\;\text{sg}[\hat{\mathbf{x}}_\text{long}],\;\log\sigma_\text{long}\big)
+
\mathcal{L}_\text{NLL}\!\big(\text{sg}[\text{target}],\;\text{sg}[\hat{\mathbf{x}}_\text{short}],\;\log\sigma_\text{short}\big)
$$

where $\mathcal{L}_\text{NLL}$ is the isotropic Gaussian negative log-likelihood:

$$
\mathcal{L}_\text{NLL}(\mathbf{x}_\star, \boldsymbol{\mu}, \sigma) =
\frac{\|\mathbf{x}_\star - \boldsymbol{\mu}\|^2}{2\,\sigma^2} + \frac{d}{2}\,\log\sigma^2,
\qquad d = C \times H \times W
$$

At optimum, $\sigma^2_\star = \frac{1}{d}\,\mathbb{E}[\|\mathbf{x}_\star - \boldsymbol{\mu}\|^2]$ : the predicted $\sigma$ learns to match the effective quadratic error of the mean prediction.

| Terme | Type | Forward | Cible | Gradient |
|-------|------|---------|-------|----------|
| $L_\text{pair}$ | MSE | $g_s(h_t, t, 0)$ | $\mathrm{sg}[g_T(h_{t'}, t', 0)]$ | $\boldsymbol{\mu}$ uniquement |
| $L_\text{long}$ | MSE | $g_s(h_t, t, 0)$ *(partagé)* | $\mathbf{x}_0$ | $\boldsymbol{\mu}$ uniquement |
| $L_\text{short}$ | MSE | $g_s(x_\text{mid}, t', 0)$ | $\mathrm{sg}[g_T(h_{t'}, t', 0)]$ | $\boldsymbol{\mu}$ uniquement |
| $L_\sigma$ | NLL | *(stop-grad $\boldsymbol{\mu}$)* | $\mathbf{x}_0$ + cible pairwise | $\alpha(\mathbf{x})$ uniquement |
| $L_\text{ae}$ | MSE | $\Phi(\mathbf{x}_0)$ | $\mathbf{x}_0$ | prior $\Phi$ uniquement |

### 2.5 Sampling SDE — bruit à chaque étape

$$
\mathbf{x}_{t_{i+1}} = c_\text{skip}(t_i,t_{i+1})\,\mathbf{x}_{t_i}
+ c_\text{out}(t_i,t_{i+1})\,\boldsymbol{\mu}_\phi(\mathbf{x}_{t_i})
+ \sigma_\phi(\mathbf{x}_{t_i}, t_i)\,\boldsymbol{\varepsilon}_i,
\quad \boldsymbol{\varepsilon}_i \sim \mathcal{N}(0,\mathbf{I})
$$

$\sigma_\phi(t_i;\mathbf{x}) = \sigma_\text{noise}\cdot t_i^{\,\alpha(\mathbf{x})}$ décroît naturellement de $\sigma_\text{noise}$ à $0$ au fil du débruitage.

| Mode | Diversité | Source |
|------|-----------|--------|
| **ODE** | $\boldsymbol{\mu}_\phi(\mathbf{x}_t)$ seulement | Init $\mathbf{x}_{t=1}=\sigma_\text{noise}\,\boldsymbol{\varepsilon}$ différente par membre |
| **SDE** | $\boldsymbol{\mu}_\phi + \sigma_\phi\,\boldsymbol{\varepsilon}_i$ à chaque pas | Diversité amplifiée à chaque étape |

### 2.6 Comparaison des paramétrisations de covariance

| Type | Forme | Paramètre appris | Implémenté |
|------|-------|-----------------|------------|
| **Loi de puissance** | $\sigma(t;\mathbf{x}) = \sigma_\text{noise}\cdot t^{\alpha(\mathbf{x})}$ | $\alpha(\mathbf{x})$ scalaire | ✓ **ici** |
| Isotrope libre | $\sigma(\mathbf{x},t)$ quelconque | $\log\sigma$ MLP | alternative |
| Diagonale | carte spatiale $\sigma_i(\mathbf{x},t)$ | $d$ valeurs | — |
| Structurée (SPDE) | $\boldsymbol{\Sigma}^{-1} = \alpha_\phi\,\mathcal{L}_\phi$ | opérateur précision | — |

---

In [ ]:
!nvidia-smi

In [ ]:
%env CUDA_VISIBLE_DEVICES=2

In [ ]:
import json
import math
import os
import zipfile
import glob
import importlib
from dataclasses import asdict, dataclass
from typing import Any, Callable, List, Optional, Tuple, Union

import numpy as np
import xarray as xr
import pandas as pd
import torch
from einops import rearrange
from einops.layers.torch import Rearrange
import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer, seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from matplotlib import pyplot as plt
from torch import Tensor, nn
from torch.nn import functional as F
from torchinfo import summary

import sys
sys.path.append('../..')    # -> consistency/
sys.path.append('../../..')  # -> 4dvarnet-starter-devs/

import consistency_models.consistency_models_VarCM as _cm_var
importlib.reload(_cm_var)

from consistency_models.consistency_models_VarCM import (
    DEFAULT_SCHEDULE,
    LinearSchedule,
)
from consistency_models.utils import karras_schedule, timesteps_schedule, update_ema_model_
from consistency_models.consistency_models_CM import ema_decay_rate_schedule

try:
    from properscoring import crps_ensemble
    HAS_PROPERSCORING = True
except ImportError:
    HAS_PROPERSCORING = False
    print("[WARNING] properscoring not installed -- CRPS disabled. Install with: pip install properscoring")


## Implementation

### DataModule — SPDE Diffusion Dataset


In [ ]:
from src.dataloader_SPDE import SPDEDataModule

SPDE_PATH = "../../../data/SPDE_diffusion_dataset_1.nc"

ds_inspect = xr.open_dataset(SPDE_PATH)
print(ds_inspect)
ds_inspect.close()

datamodule = SPDEDataModule(
    path=SPDE_PATH,
    window_size=5,
    stride=1,
    train_ratio=0.7,
    val_ratio=0.15,
    dl_kw={"batch_size": 1, "num_workers": 1},
)
datamodule.setup()
C = datamodule.window_size
print(f"window_size (C) = {C}")


In [ ]:
sample = datamodule.train_ds[0]
print(f"TrainingItem shapes -- input (y): {sample.input.shape}, tgt (x): {sample.tgt.shape}")

fig, axes = plt.subplots(1, C, figsize=(3 * C, 3))
for t_idx, ax in enumerate(axes):
    ax.imshow(sample.tgt[t_idx], origin="lower", cmap="RdBu_r")
    ax.set_title(f"x GT -- t={t_idx}", fontsize=8)
    ax.axis("off")
plt.suptitle("Training sample -- ground truth x", y=1.01)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, C, figsize=(3 * C, 3))
for t_idx, ax in enumerate(axes):
    ax.imshow(sample.input[t_idx], origin="lower", cmap="RdBu_r")
    ax.set_title(f"y obs -- t={t_idx}", fontsize=8)
    ax.axis("off")
plt.suptitle("Training sample -- observations y (NaN = unobserved)", y=1.01)
plt.tight_layout()
plt.show()


### UNet Building Blocks

GroupNorm, SelfAttention, UNetBlock, Downsample, Upsample, `TimeEmbedding` (pour **t** et **t'** séparément), `_pad_to_multiple`, `_unpad`.

*(Identique au notebook VarCM déterministe.)*


In [ ]:
def GroupNorm(channels: int) -> nn.GroupNorm:
    return nn.GroupNorm(num_groups=min(32, channels // 4), num_channels=channels)


class SelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.dropout = dropout
        self.qkv_projection = nn.Sequential(
            GroupNorm(in_channels),
            nn.Conv2d(in_channels, 3 * in_channels, kernel_size=1, bias=False),
            Rearrange("b (i h d) x y -> i b h (x y) d", i=3, h=n_heads),
        )
        self.output_projection = nn.Sequential(
            Rearrange("b h l d -> b l (h d)"),
            nn.Linear(in_channels, out_channels, bias=False),
            Rearrange("b l d -> b d l"),
            GroupNorm(out_channels),
            nn.Dropout1d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        q, k, v = self.qkv_projection(x).unbind(dim=0)
        output = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False
        )
        output = self.output_projection(output)
        output = rearrange(output, "b c (x y) -> b c x y", x=x.shape[-2], y=x.shape[-1])
        return output + self.residual_projection(x)


class UNetBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 time_level_channels: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.input_projection = nn.Sequential(
            GroupNorm(in_channels), nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.time_level_projection = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(time_level_channels, out_channels, kernel_size=1),
        )
        self.output_projection = nn.Sequential(
            GroupNorm(out_channels), nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding="same"),
            nn.Dropout2d(dropout),
        )
        self.residual_projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: Tensor, time_level: Tensor) -> Tensor:
        h = self.input_projection(x)
        h = h + self.time_level_projection(time_level)
        return self.output_projection(h) + self.residual_projection(x)


class UNetBlockWithSelfAttention(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 time_level_channels: int, n_heads: int = 8, dropout: float = 0.3) -> None:
        super().__init__()
        self.unet_block = UNetBlock(in_channels, out_channels, time_level_channels, dropout)
        self.self_attention = SelfAttention(out_channels, out_channels, n_heads, dropout)

    def forward(self, x: Tensor, time_level: Tensor) -> Tensor:
        return self.self_attention(self.unet_block(x, time_level))


class Downsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            Rearrange("b c (h ph) (w pw) -> b (c ph pw) h w", ph=2, pw=2),
            nn.Conv2d(4 * channels, channels, kernel_size=1),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


class Upsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Sequential(
            nn.Upsample(scale_factor=2.0, mode="nearest"),
            nn.Conv2d(channels, channels, kernel_size=3, padding="same"),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.projection(x)


class TimeEmbedding(nn.Module):
    """Fourier time embedding for a scalar t in [0, 1]."""
    def __init__(self, channels: int, scale: float = 16.0) -> None:
        super().__init__()
        self.W = nn.Parameter(torch.randn(channels // 2) * scale, requires_grad=False)
        self.projection = nn.Sequential(
            nn.Linear(channels, 4 * channels),
            nn.SiLU(),
            nn.Linear(4 * channels, channels),
            Rearrange("b c -> b c () ()"),
        )

    def forward(self, x: Tensor) -> Tensor:
        h = x[:, None] * self.W[None, :] * 2 * torch.pi
        h = torch.cat([torch.sin(h), torch.cos(h)], dim=-1)
        return self.projection(h)


def _pad_to_multiple(x: Tensor, multiple: int = 8) -> Tuple[Tensor, Tuple[int, int, int, int]]:
    _, _, H, W = x.shape
    pad_h = (multiple - H % multiple) % multiple
    pad_w = (multiple - W % multiple) % multiple
    padding = (0, pad_w, 0, pad_h)
    return F.pad(x, padding, mode="reflect"), padding


def _unpad(x: Tensor, padding: Tuple[int, int, int, int]) -> Tensor:
    _, pad_w, _, pad_h = padding
    H, W = x.shape[-2], x.shape[-1]
    return x[..., :H - pad_h if pad_h else H, :W - pad_w if pad_w else W]


### Stochastic Variational UNet — $\sigma(\mathbf{x}, t)$ en loi de puissance

**Différence architecturale clé vs VarCM déterministe :**

Le UNet apprend l'**exposant $\alpha(\mathbf{x})$** d'une loi de puissance, ce qui garantit structurellement les conditions aux bords :

$$
\sigma_\phi(\mathbf{x}_t, t) = \sigma_\text{noise} \cdot t^{\,\alpha(\mathbf{x}_t)},
\qquad \alpha(\mathbf{x}) = \text{softplus}\!\bigl(\text{MLP}(\text{AvgPool}(h_\text{dec}))\bigr) + \alpha_\text{min}
$$

- $t = 1$ (bruit pur) $\;\Rightarrow\; \sigma = \sigma_\text{noise}$ ✓ garanti
- $t = 0$ (débruité) $\;\Rightarrow\; \sigma = 0$ ✓ garanti
- $\alpha < 1$ : croissance concave (type hyperbolique)
- $\alpha = 1$ : linéaire
- $\alpha > 1$ : croissance convexe (type parabolique)

**Mode `stoch=True` :**
- `forward()` retourne `(mu, alpha_raw)` : `(B,C,H,W)` et `(B,1)`
- `alpha_pool` : `AdaptiveAvgPool2d(1) → Flatten` — résume $h_\text{dec}$
- `alpha_head` : `Linear(top, top//4) → SiLU → Linear(top//4, 1)` — prédit $\alpha_\text{raw}$
- Init biais à 0.541 : $\alpha_\text{init} = \text{softplus}(0.541) + 0.1 \approx 1.0$ (linéaire au départ)


In [ ]:
@dataclass
class UNetConfig:
    channels: int = 5
    time_level_channels: int = 128
    time_level_scale: float = 16.0
    n_heads: int = 8
    top_blocks_channels: Tuple[int, ...] = (64, 64)
    top_blocks_n_blocks_per_resolution: Tuple[int, ...] = (2, 2)
    top_blocks_has_resampling: Tuple[bool, ...] = (True, True)
    top_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)
    mid_blocks_channels: Tuple[int, ...] = (128, 256)
    mid_blocks_n_blocks_per_resolution: Tuple[int, ...] = (4, 4)
    mid_blocks_has_resampling: Tuple[bool, ...] = (True, False)
    mid_blocks_dropout: Tuple[float, ...] = (0.0, 0.0)
    stoch: bool = False   # If True: outputs (mu, log_sigma(x,t))


class UNet(nn.Module):
    """
    Stochastic Variational UNet: takes (c_in*x, cond, t, t') as input.

    Conditioning modes (auto-detected via cond.shape[1]):
      C   channels -> grad mode  : grad_j_projection
      2C  channels -> obs  mode  : obs_projection
      3C  channels -> obs+grad   : obs_projection + grad_j_projection

    stoch=True -> forward() returns (mu, alpha_raw) where:
      mu        : (B, C, H, W)  predicted mean
      alpha_raw : (B, 1)        unconstrained exponent for the power law
                                Full: sigma(t;x) = sigma_noise * t**alpha(x)
                                      alpha(x) = softplus(alpha_raw) + 0.1
    """
    def __init__(self, config: UNetConfig) -> None:
        super().__init__()
        self.config = config
        C   = config.channels
        top = config.top_blocks_channels[0]
        tlc = config.time_level_channels

        self.input_projection  = nn.Conv2d(C,     top, kernel_size=3, padding="same")
        self.grad_j_projection = nn.Conv2d(C,     top, kernel_size=3, padding="same")
        self.obs_projection    = nn.Conv2d(2 * C, top, kernel_size=3, padding="same")
        self.time_embedding_t  = TimeEmbedding(tlc, config.time_level_scale)
        self.time_embedding_tp = TimeEmbedding(tlc, config.time_level_scale)

        self.top_encoder_blocks = self._make_encoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout, self._make_top_block,
        )
        self.mid_encoder_blocks = self._make_encoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout, self._make_mid_block,
        )
        self.mid_decoder_blocks = self._make_decoder_blocks(
            config.mid_blocks_channels + config.mid_blocks_channels[-1:],
            config.mid_blocks_n_blocks_per_resolution,
            config.mid_blocks_has_resampling,
            config.mid_blocks_dropout, self._make_mid_block,
        )
        self.top_decoder_blocks = self._make_decoder_blocks(
            config.top_blocks_channels + config.mid_blocks_channels[:1],
            config.top_blocks_n_blocks_per_resolution,
            config.top_blocks_has_resampling,
            config.top_blocks_dropout, self._make_top_block,
        )
        self.output_projection = nn.Conv2d(top, C, kernel_size=3, padding="same")

        # ── Stochastic head: power-law exponent alpha(x) ───────────────────────
        # sigma(t; x) = sigma_noise * t**alpha(x), alpha = softplus(alpha_raw) + 0.1
        # Guarantees sigma(0)=0 and sigma(1)=sigma_noise for any alpha_raw.
        if config.stoch:
            self.alpha_pool = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),   # (B, top, H', W') -> (B, top, 1, 1)
                nn.Flatten(),              # -> (B, top)
            )
            self.alpha_head = nn.Sequential(
                nn.Linear(top, top // 4),  # pool(h_dec) only — no time embed needed
                nn.SiLU(),
                nn.Linear(top // 4, 1),    # -> (B, 1)  alpha_raw (unconstrained)
            )
            # Init bias → softplus(0.541) + 0.1 ≈ 1.0 → linear profile at start
            nn.init.zeros_(self.alpha_head[-1].weight)
            nn.init.constant_(self.alpha_head[-1].bias, 0.541)

    def forward(self, x: Tensor, cond: Tensor, t: Tensor, t_prime: Tensor
                ) -> Union[Tensor, Tuple[Tensor, Tensor]]:
        """
        x    : (B, C, H, W)
        cond : (B, C, H, W)    grad mode
             | (B, 2C, H, W)   obs mode
             | (B, 3C, H, W)   obs+grad mode
        t, t_prime : (B,)  solver times in [0, 1]

        Returns:
          stoch=False : out          (B, C, H, W)
          stoch=True  : (mu, log_s)  (B,C,H,W), (B,1)
        """
        x,    px = _pad_to_multiple(x,    multiple=8)
        cond, _  = _pad_to_multiple(cond, multiple=8)

        C = self.config.channels
        if cond.shape[1] == C:
            cond_proj = self.grad_j_projection(cond)
        elif cond.shape[1] == 2 * C:
            cond_proj = self.obs_projection(cond)
        else:  # 3C: obs+grad
            cond_proj = (self.obs_projection(cond[:, :2 * C])
                         + self.grad_j_projection(cond[:, 2 * C:]))

        h = self.input_projection(x) + cond_proj

        # Two time embeddings: t (source) and t' (target)
        emb_t  = self.time_embedding_t(t)    # (B, TLC, 1, 1) -- kept for sigma head
        emb_tp = self.time_embedding_tp(t_prime)
        time_level = torch.cat([emb_t, emb_tp], dim=1)   # (B, 2*TLC, 1, 1)

        top_encoder_embeddings = []
        for block in self.top_encoder_blocks:
            if isinstance(block, UNetBlock):
                h = block(h, time_level)
                top_encoder_embeddings.append(h)
            else:
                h = block(h)

        mid_encoder_embeddings = []
        for block in self.mid_encoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = block(h, time_level)
                mid_encoder_embeddings.append(h)
            else:
                h = block(h)

        for block in self.mid_decoder_blocks:
            if isinstance(block, UNetBlockWithSelfAttention):
                h = torch.cat((h, mid_encoder_embeddings.pop()), dim=1)
                h = block(h, time_level)
            else:
                h = block(h)

        for block in self.top_decoder_blocks:
            if isinstance(block, UNetBlock):
                h = torch.cat((h, top_encoder_embeddings.pop()), dim=1)
                h = block(h, time_level)
            else:
                h = block(h)

        out = self.output_projection(h)
        out = _unpad(out, px)

        if self.config.stoch:
            # h: (B, top, H', W')  -- top-decoder features (with padding)
            feat      = self.alpha_pool(h)    # (B, top)
            alpha_raw = self.alpha_head(feat) # (B, 1) unconstrained exponent
            return out, alpha_raw

        return out

    def _make_encoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (ic, oc) in enumerate(zip(channels[:-1], channels[1:])):
            for _ in range(n_blocks[idx]):
                blocks.append(block_fn(ic, oc, dropout[idx]))
                ic = oc
            if has_resampling[idx]:
                blocks.append(Downsample(oc))
        return blocks

    def _make_decoder_blocks(self, channels, n_blocks, has_resampling, dropout, block_fn):
        blocks = nn.ModuleList()
        for idx, (oc, ic) in enumerate(list(zip(channels[:-1], channels[1:]))[::-1]):
            if has_resampling[::-1][idx]:
                blocks.append(Upsample(ic))
            inner = []
            for _ in range(n_blocks[::-1][idx]):
                inner.append(block_fn(ic * 2, oc, dropout[::-1][idx]))
                oc = ic
            blocks.extend(inner[::-1])
        return blocks

    def _make_top_block(self, ic, oc, dropout):
        return UNetBlock(ic, oc, 2 * self.config.time_level_channels, dropout)

    def _make_mid_block(self, ic, oc, dropout):
        return UNetBlockWithSelfAttention(
            ic, oc, 2 * self.config.time_level_channels, self.config.n_heads, dropout
        )

    def save_pretrained(self, pretrained_path: str) -> None:
        os.makedirs(pretrained_path, exist_ok=True)
        with open(os.path.join(pretrained_path, "config.json"), mode="w") as f:
            json.dump(asdict(self.config), f)
        torch.save(self.state_dict(), os.path.join(pretrained_path, "model.pt"))

    @classmethod
    def from_pretrained(cls, pretrained_path: str) -> "UNet":
        with open(os.path.join(pretrained_path, "config.json"), mode="r") as f:
            config_dict = json.load(f)
        model = cls(UNetConfig(**config_dict))
        model.load_state_dict(
            torch.load(os.path.join(pretrained_path, "model.pt"), map_location="cpu")
        )
        return model


# ── Sanity check: stoch=True ─────────────────────────────────────────────────
C = datamodule.window_size
_unet_s  = UNet(UNetConfig(channels=C, stoch=True))
_x_dummy = torch.randn(2, C, 100, 100)
_t_dummy = torch.rand(2)
with torch.no_grad():
    _mu, _ar = _unet_s(_x_dummy, torch.randn(2, C, 100, 100), _t_dummy, _t_dummy)
print(f"[OK] UNet stoch=True, grad mode : mu {_mu.shape}  alpha_raw {_ar.shape}")
with torch.no_grad():
    _mu, _ar = _unet_s(_x_dummy, torch.randn(2, 2*C, 100, 100), _t_dummy, _t_dummy * 0.5)
print(f"[OK] UNet stoch=True, obs  mode : mu {_mu.shape}  alpha_raw {_ar.shape}")
# init check: alpha should be ≈1.0 (linear profile) at init
_alpha_init = (F.softplus(_ar) + 0.1).mean().item()
print(f"[CHECK] alpha at init: {_alpha_init:.4f}  (expected ≈ 1.0 — linear profile)")
_sigma_noise = 10.0
with torch.no_grad():
    _, _ar_low  = _unet_s(_x_dummy, torch.randn(2, C, 100, 100), torch.full((2,), 0.1), torch.zeros(2))
    _, _ar_high = _unet_s(_x_dummy, torch.randn(2, C, 100, 100), torch.full((2,), 0.9), torch.zeros(2))
_al = (F.softplus(_ar_low)  + 0.1).mean().item()
_ah = (F.softplus(_ar_high) + 0.1).mean().item()
print(f"[CHECK] sigma(t=0.1) ≈ {_sigma_noise * 0.1**_al:.4f}  (linear ref: {_sigma_noise*0.1:.4f})")
print(f"[CHECK] sigma(t=0.9) ≈ {_sigma_noise * 0.9**_ah:.4f}  (linear ref: {_sigma_noise*0.9:.4f})")
del _unet_s, _x_dummy, _t_dummy, _mu, _ar, _ar_low, _ar_high


### Variational Costs

- **`BaseObsCost`** : $J_o(x, y) = \|M(x - y)\|^2$ (masque NaN)
- **`BilinAEPriorCost`** : $J_b(x) = \|x - \Phi(x)\|^2$ avec $\Phi$ = bilinear auto-encoder
- **`lambda_reg`** : `nn.Parameter` scalaire $\geq 0$ pondérant $J_b$ dans $J = J_o + \lambda J_b$


In [ ]:
class BaseObsCost(nn.Module):
    """J_o(x, y) = ||M*(x - y)||^2 / (2 * sigma_obs^2), M = NaN mask."""
    def __init__(self, sigma_obs: float = 1.0) -> None:
        super().__init__()
        self.sigma_obs = sigma_obs

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        mask = (~torch.isnan(y)).to(dtype=x.dtype)
        y_c  = torch.nan_to_num(y, nan=0.0)
        diff = mask * (x - y_c)
        return (diff ** 2).sum() / (2.0 * self.sigma_obs ** 2)


class BilinAEPriorCost(nn.Module):
    """Prior cost: J_b(x) = ||x - Phi(x)||^2, Phi = bilinear auto-encoder."""
    def __init__(self, dim_in: int = 5, dim_ae: int = 32) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(dim_in, dim_ae, 3, padding=1), nn.SiLU(),
            nn.Conv2d(dim_ae, dim_ae, 3, padding=1), nn.SiLU(),
        )
        self.decoder_lin = nn.Sequential(
            nn.Conv2d(dim_ae, dim_ae, 3, padding=1), nn.SiLU(),
            nn.Conv2d(dim_ae, dim_in, 3, padding=1),
        )
        self.decoder_quad = nn.Sequential(
            nn.Conv2d(dim_ae, dim_ae, 3, padding=1), nn.SiLU(),
            nn.Conv2d(dim_ae, dim_in, 3, padding=1),
        )

    def forward_ae(self, x: Tensor) -> Tensor:
        z = self.encoder(x)
        return self.decoder_lin(z) + self.decoder_quad(z) ** 2

    def forward(self, x: Tensor) -> Tensor:
        return F.mse_loss(x, self.forward_ae(x)) * x.numel()


# Sanity check
C = datamodule.window_size
_x  = torch.randn(2, C, 100, 100)
_y  = torch.randn(2, C, 100, 100)
_y[0, :, :10, :10] = float('nan')
_oc = BaseObsCost(sigma_obs=1.0)
_pc = BilinAEPriorCost(dim_in=C)
print(f"J_o = {_oc(_x, _y).item():.2f}")
print(f"J_b = {_pc(_x).item():.2f}")
print(f"Phi(x) shape = {_pc.forward_ae(_x).shape}")
del _x, _y, _oc, _pc


### Stochastic VarCM — Loss découplée : MSE pour $\boldsymbol{\mu}$ + NLL pour $\sigma$

#### `stoch_forward` — wrapper pour UNet stochastique

Analogue de `model_variational_forward_wrapper` (VarCM), mais le UNet retourne `(mu, alpha_raw)` :

$$
(\boldsymbol{\mu}_\phi, \alpha_\text{raw}) = F_\theta(c_\text{in}(t)\,\mathbf{x}_t,\;\mathbf{c},\;t,\;t')
$$
$$
\sigma_\phi(\mathbf{x}_t, t) = \sigma_\text{noise}\cdot t^{\,\alpha(\mathbf{x}_t)},\quad \alpha = \text{softplus}(\alpha_\text{raw}) + 0.1
$$
$$
g_\phi^\text{stoch}(\mathbf{x}_t, \mathbf{c}, t, t') = c_\text{skip}\,\mathbf{x}_t + c_\text{out}\,\boldsymbol{\mu}_\phi
\quad + \quad\sigma_\phi(\mathbf{x}_t,\,t)\,\boldsymbol{\varepsilon}\;\;\text{(SDE, à chaque étape)}
$$

#### `anchoring_nll_iso` — NLL isotrope

$$
\mathcal{L}_\text{NLL}(\mathbf{x}_\star, \boldsymbol{\mu}, \sigma) =
\frac{\|\mathbf{x}_\star - \boldsymbol{\mu}\|^2}{2\,\sigma^2(t)} + \frac{d}{2}\,\log\sigma^2(t)
$$

Implémentation efficace : $\sigma^2 = \exp(2\,\log\sigma)$, résidu sommé sur $d = C \times H \times W$, NLL moyennée sur le batch.

#### `StochasticVarCMTraining` — Loss structurée

```
h_t  = sg[teacher(x_T, T, t)]            teacher forward (no_grad)
h_t' = sg[teacher(x_T, T, t')]           teacher forward (no_grad)
target = sg[teacher(h_t', t', 0)]        teacher cible pairwise

(pred_long, log_s_long) = student(h_t, t, 0)     # forward STUDENT partagé
L_pair  = MSE(target,  pred_long)                 # saut pairwise   (= VarCM)
L_long  = MSE(x_0,     pred_long)                 # ancrage long    (= VarCM)
L_sigma_long = NLL(x_0, sg[pred_long], log_s_long)  # calibration σ (stop-grad mu)

x_mid = sg[student(h_t, t, t')]                      # sg intermédiaire
(pred_short, log_s_short) = student(x_mid, t', 0)    # forward STUDENT séparé
L_short       = MSE(target, pred_short)               # ancrage court  (= VarCM)
L_sigma_short = NLL(target, sg[pred_short], log_s_short)  # calibration σ

L_sigma = L_sigma_long + L_sigma_short
L_ae    = MSE(Phi(x_0), x_0)
```

**Points clés :**
- `L_pair`, `L_long`, `L_short` = **MSE identique au VarCM** → gradient plein sur μ → diffusion physique apprise
- `L_sigma` = NLL avec **stop-gradient sur μ** → seul `alpha_head` reçoit le gradient → calibration σ sans interférer sur μ
- Découplage essentiel : NLL pur atténue le gradient μ d'un facteur 1/σ² ≈ 1/25 à mi-chemin, empêchant l'apprentissage physique


In [ ]:
# ── stoch_forward: wrapper for stochastic UNet ───────────────────────────────

def stoch_forward(
    model:       nn.Module,
    obs_cost:    nn.Module,
    prior_cost:  nn.Module,
    lambda_reg:  nn.Parameter,
    x:           Tensor,          # (B, C, H, W)  current state
    y:           Tensor,          # (B, C, H, W)  observations
    t:           Tensor,          # (B,)  source time
    t_prime:     Tensor,          # (B,)  target time
    schedule=None,
    sigma_noise: float = 1.0,
    conditioning_mode: str = "obs",
    z:           Optional[Tensor] = None,   # noise for SDE at final step
) -> Tuple[Tensor, Tensor, Tensor]:
    """
    Returns: (x_out, log_sigma, cond)
      x_out     : c_skip*x + c_out*mu  (+ sigma*z if z provided)
      log_sigma : log sigma(x, t) -- predicted by the stochastic head
      cond      : conditioning tensor used (for inspection)
    """
    if schedule is None:
        schedule = DEFAULT_SCHEDULE

    try:
        model_dtype = next(model.parameters()).dtype
    except StopIteration:
        model_dtype = x.dtype

    B, C, H, W = x.shape

    # ── Build conditioning tensor ─────────────────────────────────────────────
    if conditioning_mode == "obs":
        mask   = (~torch.isnan(y)).to(dtype=model_dtype)
        y_fill = torch.nan_to_num(y, nan=0.0).to(dtype=model_dtype)
        cond   = torch.cat([y_fill, mask], dim=1)   # (B, 2C, H, W)

    elif conditioning_mode in ("grad", "obs+grad"):
        x_leaf = x.detach().float().requires_grad_(True)
        y_c    = y.detach().float()
        lr_f   = lambda_reg.detach().float().clamp(min=0.0)
        try:
            cost_dtype = next(prior_cost.parameters()).dtype
        except StopIteration:
            cost_dtype = torch.float32
        with torch.enable_grad():
            Jo = obs_cost(x_leaf.to(cost_dtype), y_c.to(cost_dtype))
            Jb = prior_cost(x_leaf.to(cost_dtype))
            J  = (Jo + lr_f.to(cost_dtype) * Jb).float()
            g  = torch.autograd.grad(J, x_leaf, create_graph=False)[0].detach()
        grad_J_norm = (g / g.std().clamp(min=1e-8)).to(dtype=model_dtype)

        if conditioning_mode == "obs+grad":
            mask   = (~torch.isnan(y)).to(dtype=model_dtype)
            y_fill = torch.nan_to_num(y, nan=0.0).to(dtype=model_dtype)
            cond   = torch.cat([y_fill, mask, grad_J_norm], dim=1)   # (B, 3C, H, W)
        else:
            cond = grad_J_norm   # (B, C, H, W)
    else:
        raise ValueError(f"Unknown conditioning_mode={conditioning_mode!r}")

    # ── Preconditioning + UNet forward ────────────────────────────────────────
    x_det = x.detach().to(dtype=model_dtype)

    def _expand(s: Tensor) -> Tensor:
        return s.to(dtype=model_dtype).view(B, 1, 1, 1)

    c_in   = _expand(schedule.c_in(t,   sigma_noise))
    c_skip = _expand(schedule.c_skip(t, t_prime))
    c_out  = _expand(schedule.c_out(t,  t_prime))

    F_mu, alpha_raw = model(
        c_in * x_det,
        cond,
        t.to(dtype=model_dtype),
        t_prime.to(dtype=model_dtype),
    )
    # Parametric power law: sigma(t; x) = sigma_noise * t**alpha(x)
    #   alpha(x) = softplus(alpha_raw) + 0.1  (strictly > 0.1)
    #   t=0 -> sigma=0  (clean, no uncertainty)   -- guaranteed by t**alpha -> 0
    #   t=1 -> sigma=sigma_noise                  -- guaranteed by t**alpha -> 1
    alpha = (F.softplus(alpha_raw) + 0.1).view(B, 1)                        # (B,1)
    log_t = torch.log(t.to(dtype=model_dtype).clamp(min=1e-8)).view(B, 1)   # (B,1)
    log_sigma = math.log(sigma_noise) + alpha * log_t                        # (B,1)

    x_mean = c_skip * x_det + c_out * F_mu

    if z is not None:
        sigma = torch.exp(log_sigma).view(B, 1, 1, 1)
        x_out = x_mean + sigma * z.to(dtype=model_dtype)
    else:
        x_out = x_mean

    return x_out, log_sigma, cond


# ── NLL isotrope ─────────────────────────────────────────────────────────────

def anchoring_nll_iso(
    x_target:  Tensor,   # (B, C, H, W)  target
    mu:        Tensor,   # (B, C, H, W)  predicted mean
    log_sigma: Tensor,   # (B, 1)        scalar log sigma per sample
) -> Tensor:
    """
    Isotropic Gaussian NLL:
        L = ||x_target - mu||^2 / (2*sigma^2)  +  (d/2) * log(sigma^2)
    Optimal at sigma^2 = (1/d) * E[||x_target - mu||^2]
    """
    B, C, H, W = x_target.shape
    d        = C * H * W
    sigma2   = torch.exp(2.0 * log_sigma).squeeze(1)              # (B,)
    residual = (x_target - mu).pow(2).reshape(B, -1).sum(1)       # (B,) sum over d
    nll      = residual / (2.0 * sigma2 + 1e-8) + (d / 2.0) * torch.log(sigma2 + 1e-8)
    return nll.mean() / d   # normalise par element — meme echelle que F.mse_loss


# ── StochasticVarCMTraining ───────────────────────────────────────────────────

@dataclass
class StochVarCMOutput:
    loss:          Tensor
    l_pair:        Tensor   # MSE: g_s(h_t, t, 0) vs target_pair   (mean, like VarCM)
    l_anc:         Tensor   # MSE: L_long + L_short                  (mean, like VarCM)
    l_sigma:       Tensor   # NLL: sigma calibration (stop-grad mu)  (sigma only)
    l_ae:          Tensor   # MSE: AE reconstruction
    l_long:        Tensor   # MSE: g_s(h_t, t, 0) vs x_0
    l_short:       Tensor   # MSE: g_s(x_mid, t', 0) vs target_pair
    num_timesteps: int
    times:         Tensor
    t_sampled:     Tensor
    t_p_sampled:   Tensor


class StochasticVarCMTraining(nn.Module):
    """
    Stochastic VarCM training with togglable stochastic mode.

    stochastic=False : L_pair + L_long + L_short + L_ae  (MSE, IDENTICAL to VarCM)
    stochastic=True  : adds L_sigma = NLL(stop-grad mu) to calibrate alpha_head only

    Key design choices:
      - MSE terms always present => strong gradient on mu, physical diffusion learned
      - L_sigma isolated by stop-gradient on mu => alpha_head trains independently
      - stochastic=False lets you verify deterministic convergence matches VarCM exactly
    """

    def __init__(
        self,
        initial_timesteps: int   = 5,
        final_timesteps:   int   = 50,
        sigma_noise:       float = 10.0,
        lambda_pair:       float = 1.0,
        lambda_anc:        float = 1.0,
        lambda_sigma:      float = 1.0,
        lambda_ae:         float = 1.0,
        schedule_power:    float = 1.0,
        conditioning_mode: str   = "obs",
        pure_short:        bool  = True,
        stochastic:        bool  = True,
    ) -> None:
        super().__init__()
        self.initial_timesteps = initial_timesteps
        self.final_timesteps   = final_timesteps
        self.sigma_noise       = sigma_noise
        self.lambda_pair       = lambda_pair
        self.lambda_anc        = lambda_anc
        self.lambda_sigma      = lambda_sigma
        self.lambda_ae         = lambda_ae
        self.schedule_power    = schedule_power
        self.conditioning_mode = conditioning_mode
        self.pure_short        = pure_short
        self.stochastic        = stochastic

    def _fwd(self, model, obs_cost, prior_cost, lambda_reg, x, y, t, t_prime):
        """Convenience wrapper: returns (x_out, log_sigma)."""
        return stoch_forward(
            model, obs_cost, prior_cost, lambda_reg, x, y, t, t_prime,
            sigma_noise=self.sigma_noise,
            conditioning_mode=self.conditioning_mode,
        )[:2]

    def forward(
        self,
        student:     nn.Module,
        teacher:     nn.Module,
        obs_cost:    nn.Module,
        prior_cost:  nn.Module,
        lambda_reg:  nn.Parameter,
        x0:          Tensor,
        y:           Tensor,
        global_step: int,
        total_steps: int,
    ) -> StochVarCMOutput:
        device = x0.device
        B      = x0.shape[0]

        # ── Time schedule ──────────────────────────────────────────────────────
        N = timesteps_schedule(global_step, total_steps,
                               self.initial_timesteps, self.final_timesteps)
        times = karras_schedule(N, sigma_min=0.002 / self.sigma_noise,
                                sigma_max=1.0, rho=7.0, device=device)
        times = times.flip(0).pow(self.schedule_power).clamp(0.0, 1.0)
        if times[-1].item() > 1e-3:
            times = torch.cat([times, torch.zeros(1, device=device)])

        T_val = times[0].item()
        eps   = 1e-4
        n     = len(times)

        interior = max(n - 2, 2)
        pair_idx = torch.randperm(interior, device=device)[:2].sort().values
        j_idx, i_idx = int(pair_idx[0]), int(pair_idx[1])
        t_s    = times[j_idx + 1].expand(B)   # source time (larger)
        t_p    = times[i_idx + 1].expand(B)   # target time (smaller)
        T_vec  = torch.full((B,), T_val, device=device, dtype=x0.dtype)
        t0_vec = torch.full((B,), eps,   device=device, dtype=x0.dtype)

        # ── Teacher forwards (no_grad) ─────────────────────────────────────────
        x_T = torch.randn_like(x0) * self.sigma_noise
        with torch.no_grad():
            h_t,  _        = self._fwd(teacher, obs_cost, prior_cost, lambda_reg,
                                       x_T, y, T_vec, t_s)
            h_tp, _        = self._fwd(teacher, obs_cost, prior_cost, lambda_reg,
                                       x_T, y, T_vec, t_p)
            target_pair, _ = self._fwd(teacher, obs_cost, prior_cost, lambda_reg,
                                       h_tp, y, t_p, t0_vec)

        # ── SHARED student forward: h_t -> 0 (used for L_pair AND L_long) ─────
        pred_long, log_s_long = self._fwd(student, obs_cost, prior_cost, lambda_reg,
                                          h_t, y, t_s, t0_vec)
        # Mean losses — MSE identical to VarCM in both stochastic modes
        l_pair = F.mse_loss(pred_long, target_pair.detach())
        l_long = F.mse_loss(pred_long, x0.to(pred_long.dtype))

        # ── Short anchor: student(x_mid, t', 0) ───────────────────────────────
        with torch.no_grad():
            x_mid, _ = self._fwd(student, obs_cost, prior_cost, lambda_reg,
                                  h_t, y, t_s, t_p)
        pred_short, log_s_short = self._fwd(student, obs_cost, prior_cost, lambda_reg,
                                             x_mid.detach(), y, t_p, t0_vec)
        l_short = F.mse_loss(pred_short, target_pair.detach())

        # ── AE loss (MSE, trains prior_cost) ──────────────────────────────────
        try:
            cost_dtype = next(prior_cost.parameters()).dtype
        except StopIteration:
            cost_dtype = torch.float32
        x_gt = x0.to(dtype=cost_dtype)
        l_ae = F.mse_loss(prior_cost.forward_ae(x_gt), x_gt).float()

        l_anc = l_long + l_short

        # ── Sigma calibration (stochastic=True only) ──────────────────────────
        # Stop-gradient on mu: only alpha_head receives gradient from l_sigma
        if self.stochastic:
            l_sigma = (
                anchoring_nll_iso(x0.to(pred_long.dtype), pred_long.detach(), log_s_long)
                + anchoring_nll_iso(target_pair.detach(), pred_short.detach(), log_s_short)
            )
            loss = (self.lambda_pair  * l_pair
                  + self.lambda_anc   * l_anc
                  + self.lambda_sigma * l_sigma
                  + self.lambda_ae    * l_ae)
        else:
            # Pure MSE — identical to VarCM
            l_sigma = torch.zeros(1, device=x0.device).squeeze()
            loss    = (self.lambda_pair * l_pair
                     + self.lambda_anc  * l_anc
                     + self.lambda_ae   * l_ae)

        return StochVarCMOutput(
            loss=loss, l_pair=l_pair, l_anc=l_anc, l_sigma=l_sigma, l_ae=l_ae,
            l_long=l_long, l_short=l_short,
            num_timesteps=N, times=times,
            t_sampled=t_s, t_p_sampled=t_p,
        )


# ── Sanity check ──────────────────────────────────────────────────────────────
C = datamodule.window_size
_stoch_ct = StochasticVarCMTraining(
    initial_timesteps=3, final_timesteps=5, sigma_noise=10.0,
    lambda_pair=1.0, lambda_anc=1.0, lambda_sigma=1.0, lambda_ae=0.5,
    conditioning_mode="obs", pure_short=True,
)
_unet_s   = UNet(UNetConfig(channels=C, stoch=True))
_teacher  = UNet(UNetConfig(channels=C, stoch=True))
_teacher.load_state_dict(_unet_s.state_dict())
_obs   = BaseObsCost()
_prior = BilinAEPriorCost(dim_in=C)
_lr    = nn.Parameter(torch.tensor(1.0))
_batch = next(iter(datamodule.train_dataloader()))
with torch.no_grad():
    _out = _stoch_ct(_unet_s, _teacher, _obs, _prior, _lr,
                     _batch.tgt, _batch.input, global_step=0, total_steps=5000)
for _mode in [False, True]:
    _stoch_ct.stochastic = _mode
    with torch.no_grad():
        _out = _stoch_ct(_unet_s, _teacher, _obs, _prior, _lr,
                         _batch.tgt, _batch.input, global_step=0, total_steps=5000)
    _label = "stoch=True " if _mode else "stoch=False"
    print(
        f"[OK] {_label}  loss={_out.loss.item():.4f}  N={_out.num_timesteps}\n"
        f"  L_pair={_out.l_pair.item():.4f}  L_anc={_out.l_anc.item():.4f}  "
        f"(L_long={_out.l_long.item():.4f}  L_short={_out.l_short.item():.4f})  "
        f"L_sigma={_out.l_sigma.item():.6f}  L_ae={_out.l_ae.item():.4f}"
    )
del _unet_s, _teacher, _obs, _prior, _lr, _batch, _out, _stoch_ct


In [ ]:
@dataclass
class LitStochVarCMConfig:
    initial_ema_decay_rate:       float = 0.95
    student_model_ema_decay_rate: float = 0.99993
    lr:                           float = 1e-4
    betas:             Tuple[float, float] = (0.9, 0.995)
    lr_scheduler_start_factor:    float = 1e-5
    lr_scheduler_iters:           int   = 10_000
    lambda_reg_init:              float = 1.0
    total_training_steps:         int   = 5_000
    lambda_sigma:                 float = 1.0


class LitStochasticVarCM(LightningModule):
    """
    Lightning wrapper for Stochastic VarCM (isotropic sigma conditioned on t).

    Logs: train_loss, L_pair_nll, L_anc_nll, L_long_nll, L_short_nll, L_ae,
          num_timesteps, lambda_reg, ema_decay_rate.
    """

    def __init__(
        self,
        stoch_training:    StochasticVarCMTraining,
        student_model:     UNet,
        teacher_model:     UNet,
        ema_student_model: UNet,
        obs_cost:          BaseObsCost,
        prior_cost:        BilinAEPriorCost,
        config:            LitStochVarCMConfig,
    ) -> None:
        super().__init__()
        self.stoch_training    = stoch_training
        self.student_model     = student_model
        self.teacher_model     = teacher_model
        self.ema_student_model = ema_student_model
        self.obs_cost          = obs_cost
        self.prior_cost        = prior_cost
        self.config            = config
        self.num_timesteps     = stoch_training.initial_timesteps

        self.lambda_reg = nn.Parameter(
            torch.tensor(config.lambda_reg_init, dtype=torch.float32)
        )
        for p in self.teacher_model.parameters():
            p.requires_grad = False
        for p in self.ema_student_model.parameters():
            p.requires_grad = False
        self.teacher_model.eval()
        self.ema_student_model.eval()

    def on_train_epoch_start(self) -> None:
        print(f"[Epoch {self.current_epoch}] N={self.num_timesteps}  "
              f"lr={self.optimizers().param_groups[0]['lr']:.2e}  "
              f"lambda_reg={self.lambda_reg.item():.4f}")

    def training_step(self, batch, batch_idx: int):
        if isinstance(batch, list):
            batch = batch[0]
        self.lambda_reg.data.clamp_(min=0.0)

        out = self.stoch_training(
            self.student_model, self.teacher_model,
            self.obs_cost, self.prior_cost, self.lambda_reg,
            batch.tgt, batch.input,
            self.global_step, self._total_training_steps,
        )
        self.num_timesteps = out.num_timesteps

        if batch_idx % 20 == 0:
            _mode = "S" if self.stoch_training.stochastic else "D"
            print(
                f"  [{_mode} step {self.global_step}] N={out.num_timesteps}  "
                f"loss={out.loss.item():.4f}  "
                f"L_pair={out.l_pair.item():.4f}  "
                f"L_anc={out.l_anc.item():.4f}  "
                f"(L_long={out.l_long.item():.4f}  L_short={out.l_short.item():.4f})  "
                f"L_sigma={out.l_sigma.item():.4f}  L_ae={out.l_ae.item():.4f}"
            )

        self.log_dict({
            "train_loss":    out.loss,
            "L_pair":        out.l_pair,
            "L_anc":         out.l_anc,
            "L_long":        out.l_long,
            "L_short":       out.l_short,
            "L_sigma":       out.l_sigma,
            "L_ae":          out.l_ae,
            "num_timesteps": float(out.num_timesteps),
            "lambda_reg":    self.lambda_reg.detach(),
        }, prog_bar=False)
        return out.loss

    def on_train_batch_end(self, outputs, batch, batch_idx: int) -> None:
        ema_decay = ema_decay_rate_schedule(
            self.num_timesteps,
            self.config.initial_ema_decay_rate,
            self.stoch_training.initial_timesteps,
        )
        update_ema_model_(self.teacher_model,     self.student_model, ema_decay)
        update_ema_model_(self.ema_student_model, self.student_model,
                          self.config.student_model_ema_decay_rate)
        self.log("ema_decay_rate", ema_decay)

    def configure_optimizers(self):
        # ── Dynamic total_training_steps ─────────────────────────────────────
        # Use the actual number of optimizer steps for the full run so that
        # N grows from initial_timesteps to final_timesteps gradually across
        # all epochs — not just the first config.total_training_steps steps.
        try:
            actual_steps = self.trainer.estimated_stepping_batches
            if actual_steps and int(actual_steps) > 0:
                self._total_training_steps = int(actual_steps)
                print(
                    f"[LitStochasticVarCM] Dynamic total_training_steps = "
                    f"{self._total_training_steps}  "
                    f"(config fallback was {self.config.total_training_steps})"
                )
        except Exception as e:
            print(
                f"[LitStochasticVarCM] Could not get estimated_stepping_batches ({e}), "
                f"using config value {self.config.total_training_steps}"
            )

        params = (
            list(self.student_model.parameters())
            + list(self.obs_cost.parameters())
            + list(self.prior_cost.parameters())
            + [self.lambda_reg]
        )
        opt   = torch.optim.Adam(params, lr=self.config.lr, betas=self.config.betas)
        sched = torch.optim.lr_scheduler.LinearLR(
            opt,
            start_factor=self.config.lr_scheduler_start_factor,
            total_iters=self.config.lr_scheduler_iters,
        )
        return [opt], [{"scheduler": sched, "interval": "step", "frequency": 1}]


## Training


In [ ]:
import zipfile

def _is_valid_ckpt(path: str) -> bool:
    try:
        with zipfile.ZipFile(path): pass
        return True
    except Exception:
        return False

def find_best_valid_checkpoint(ckpt_dir: str, use_last: bool = True):
    if not os.path.isdir(ckpt_dir):
        print(f"[INFO] No checkpoint dir at {ckpt_dir!r} -- starting from scratch.")
        return None
    if use_last:
        last = os.path.join(ckpt_dir, "last.ckpt")
        if os.path.isfile(last) and _is_valid_ckpt(last):
            print(f"[RESUME] last checkpoint: {last}")
            return last
        print("[WARN] last.ckpt missing or corrupted -- falling back to best.")
    ckpts = [c for c in glob.glob(os.path.join(ckpt_dir, "*.ckpt"))
             if "last" not in os.path.basename(c) and _is_valid_ckpt(c)]
    if not ckpts:
        print(f"[INFO] No valid checkpoint in {ckpt_dir!r} -- starting from scratch.")
        return None
    best = min(ckpts,
               key=lambda p: float(p.split("train_loss=")[-1].replace(".ckpt", ""))
               if "train_loss=" in p else float("inf"))
    print(f"[RESUME] best checkpoint: {best}")
    return best

RESET_TRAINING   = False

C = datamodule.window_size

COND_MODE   = "obs"          # "obs" | "grad" | "obs+grad"
SIGMA_NOISE = 10.0

# ── Mode stochastique ─────────────────────────────────────────────────────────
# False : MSE pur identique au VarCM — vérification de convergence déterministe
# True  : ajoute L_sigma (NLL stop-grad mu) pour calibrer alpha_head
STOCHASTIC = True

LOG_DIR     = f"logs_CT_VarCM_stoch_spde_{COND_MODE.replace('+', '_')}
CKPT_DIR    = os.path.join(LOG_DIR, "checkpoints")
MODEL_PATH  = os.path.join(LOG_DIR, "best_model")

if RESET_TRAINING:
    import shutil
    for d in [CKPT_DIR, MODEL_PATH]:
        if os.path.exists(d):
            shutil.rmtree(d)
            print(f"[DELETE] {d}")
    resume_ckpt = None
else:
    resume_ckpt = find_best_valid_checkpoint(CKPT_DIR)

SCHEDULE_POWER = 1.0

stoch_ct = StochasticVarCMTraining(
    initial_timesteps = 5,
    final_timesteps   = 50,
    sigma_noise       = SIGMA_NOISE,
    lambda_pair       = 1.0,
    lambda_anc        = 1.0,
    lambda_sigma      = 1.0,
    lambda_ae         = 1.0,
    schedule_power    = SCHEDULE_POWER,
    conditioning_mode = COND_MODE,
    pure_short        = True,
    stochastic        = STOCHASTIC,
)

student_model     = UNet(UNetConfig(channels=C, stoch=True))
teacher_model     = UNet(UNetConfig(channels=C, stoch=True))
ema_student_model = UNet(UNetConfig(channels=C, stoch=True))
teacher_model.load_state_dict(student_model.state_dict())
ema_student_model.load_state_dict(student_model.state_dict())

obs_cost   = BaseObsCost(sigma_obs=1.0)
prior_cost = BilinAEPriorCost(dim_in=C)

lit_stoch = LitStochasticVarCM(
    stoch_ct,
    student_model, teacher_model, ema_student_model,
    obs_cost, prior_cost,
    LitStochVarCMConfig(lr_scheduler_iters=1000, total_training_steps=5000),
)

trainer = Trainer(
    accelerator             = "gpu",
    max_epochs              = 2500,
    accumulate_grad_batches = 4,
    precision               = "bf16-mixed",
    gradient_clip_val       = 0.5,
    gradient_clip_algorithm = "norm",
    log_every_n_steps       = 1,
    logger   = TensorBoardLogger(".", name=LOG_DIR, version=""),
    callbacks = [
        LearningRateMonitor(logging_interval="step"),
        ModelCheckpoint(
            dirpath    = CKPT_DIR,
            monitor    = "train_loss",
            save_top_k = 3,
            save_last  = True,
            filename   = "{epoch:03d}-{step}-{train_loss:.4f}",
        ),
    ],
)

seed_everything(42)
trainer.fit(lit_stoch, datamodule, ckpt_path=resume_ckpt)

# ── Save EMA UNet + costs ─────────────────────────────────────────────────────
os.makedirs(MODEL_PATH, exist_ok=True)
lit_stoch.ema_student_model.save_pretrained(MODEL_PATH)
torch.save(
    {
        "obs_cost":          lit_stoch.obs_cost.state_dict(),
        "prior_cost":        lit_stoch.prior_cost.state_dict(),
        "lambda_reg":        lit_stoch.lambda_reg.detach().cpu(),
        "schedule_power":    SCHEDULE_POWER,
        "conditioning_mode": COND_MODE,
        "sigma_noise":       SIGMA_NOISE,
    },
    os.path.join(MODEL_PATH, "costs.pt"),
)
print(f"[OK] StochVarCM EMA model saved to: {MODEL_PATH}")


## Sampling & Evaluation


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

C = datamodule.window_size

unet = UNet.from_pretrained(MODEL_PATH).eval().to(device=device, dtype=dtype)

_costs = torch.load(os.path.join(MODEL_PATH, "costs.pt"), map_location="cpu")
obs_cost   = BaseObsCost(sigma_obs=1.0).to(device=device, dtype=dtype)
prior_cost = BilinAEPriorCost(dim_in=C).to(device=device, dtype=dtype)
obs_cost.load_state_dict(_costs["obs_cost"])
prior_cost.load_state_dict(_costs["prior_cost"])
obs_cost.eval();  prior_cost.eval()

lambda_reg    = nn.Parameter(_costs["lambda_reg"].to(device=device, dtype=torch.float32))
SIGMA_NOISE   = _costs.get("sigma_noise", 10.0)
COND_MODE_LOAD = _costs.get("conditioning_mode", "obs")
SCHED_POW     = _costs.get("schedule_power", 1.0)

print(f"[OK] Model loaded from: {MODEL_PATH}")
print(f"     lambda_reg={lambda_reg.item():.4f}  "
      f"cond={COND_MODE_LOAD!r}  sigma_noise={SIGMA_NOISE}  sched_power={SCHED_POW}")


### Profil $\sigma_\phi(t; \mathbf{x})$ — Loi de puissance apprise

**Diagnostic clé du StochVarCM :** le réseau apprend un exposant $\alpha(\mathbf{x})$ tel que

$$
\sigma_\phi(\mathbf{x}_t, t) = \sigma_\text{noise} \cdot t^{\,\alpha(\mathbf{x}_t)}, \quad \alpha > 0.1
$$

**Garanties structurelles :**
- $\sigma_\phi(t=0) = 0$ (état propre, aucune incertitude) ✓ garanti
- $\sigma_\phi(t=1) = \sigma_\text{noise}$ (bruit pur, incertitude max) ✓ garanti

**Attendu après entraînement :**
- $\alpha \approx 1$ : profil linéaire (neutre)
- $\alpha < 1$ : concave — forte incertitude sur les premiers pas (début de débruitage difficile)
- $\alpha > 1$ : convexe — incertitude plus concentrée sur les $t$ élevés (bruit pur)

On sonde $\alpha(\mathbf{x})$ sur un batch test en fixant $t' = 0$.


In [ ]:
seed_everything(42)
batch = next(iter(datamodule.test_dataloader()))
H, W  = batch.tgt.shape[-2], batch.tgt.shape[-1]

# ── Probe sigma(t) at different solver times ──────────────────────────────────
# Fix t'=0 (the mapping that is supervised by the NLL loss),
# vary t from near-clean to pure-noise.

t_probe  = torch.linspace(0.02, 0.98, 60, device=device)
y_probe  = batch.input[:4].to(device=device, dtype=dtype)   # 4 samples for alpha distribution
x_noise  = torch.randn(4, C, H, W, device=device, dtype=dtype) * SIGMA_NOISE

# ── Collect alpha(x) at t=0.5 (representative intermediate step) ─────────────
alpha_vals = []   # (N_batch, 1)
sigma_vals = []   # sigma(t; x) per t
t0_vec = torch.zeros(4, device=device, dtype=dtype)

with torch.no_grad():
    for t_val in t_probe:
        t_vec  = t_val.view(1).expand(4).to(dtype=dtype)
        _, log_s, _ = stoch_forward(
            unet, obs_cost, prior_cost, lambda_reg,
            x_noise, y_probe, t_vec, t0_vec,
            sigma_noise=SIGMA_NOISE, conditioning_mode=COND_MODE_LOAD,
        )
        sigma_vals.append(torch.exp(log_s).mean().item())

# Collect alpha distribution from a full test batch
alpha_batch = []
with torch.no_grad():
    for _b in [batch.input[:16].to(device=device, dtype=dtype)]:
        _x  = torch.randn(len(_b), C, H, W, device=device, dtype=dtype) * SIGMA_NOISE
        _t  = torch.full((len(_b),), 0.5, device=device, dtype=dtype)
        _t0 = torch.zeros(len(_b), device=device, dtype=dtype)
        _mu, _ar, _ = stoch_forward(unet, obs_cost, prior_cost, lambda_reg,
                                    _x, _b, _t, _t0,
                                    sigma_noise=SIGMA_NOISE,
                                    conditioning_mode=COND_MODE_LOAD)
        _alpha = F.softplus(_ar.float()) + 0.1
        alpha_batch.append(_alpha.squeeze(1).cpu().numpy())
alpha_np = np.concatenate(alpha_batch)

t_np    = t_probe.cpu().numpy()
sig_np  = np.array(sigma_vals)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: sigma(t) curves for alpha=0.5, 1, 2 + learned mean
for _a, _c, _lbl in [(0.5, "royalblue", "α=0.5 (concave)"),
                     (1.0, "black",     "α=1.0 (linéaire)"),
                     (2.0, "seagreen",  "α=2.0 (parabolique)")]:
    axes[0].plot(t_np, SIGMA_NOISE * t_np**_a, linestyle="--", color=_c, alpha=0.6, label=_lbl)
axes[0].plot(t_np, sig_np, linewidth=2.5, color="tomato", marker="o", markersize=3, label=r"$\sigma_\phi(t)$ appris")
axes[0].axhline(y=SIGMA_NOISE, color="gray", linestyle=":", alpha=0.5, label=f"σ_noise={SIGMA_NOISE}")
axes[0].set_xlabel("Solver time $t$  [0=propre, 1=bruit pur]")
axes[0].set_ylabel(r"$\sigma_\phi(t;\mathbf{x})$")
axes[0].set_title(r"Profil $\sigma_\phi(t) = \sigma_	ext{noise}\cdot t^{\alpha(\mathbf{x})}$")
axes[0].grid(True, alpha=0.3);  axes[0].legend(fontsize=8)

# Right: alpha distribution
axes[1].hist(alpha_np, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].axvline(x=1.0, color="black",   linestyle="--", label="linéaire (α=1)")
axes[1].axvline(x=alpha_np.mean(), color="tomato", linestyle="-", lw=2,
                label=f"moyenne α={alpha_np.mean():.2f}")
axes[1].set_xlabel(r"$\alpha(\mathbf{x})$ = softplus(alpha_raw) + 0.1")
axes[1].set_ylabel("Effectif")
axes[1].set_title(r"Distribution de $\alpha(\mathbf{x})$ sur le batch test")
axes[1].legend(fontsize=8);  axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"alpha: mean={alpha_np.mean():.3f}  std={alpha_np.std():.3f}  "
      f"min={alpha_np.min():.3f}  max={alpha_np.max():.3f}")
print(f"sigma(t=0.1) = {sig_np[3]:.4f}  (alpha=1 ref: {SIGMA_NOISE*0.1:.4f})")
print(f"sigma(t=0.9) = {sig_np[-5]:.4f}  (alpha=1 ref: {SIGMA_NOISE*0.9:.4f})")

### Test Batch


In [ ]:
def plot_field(arr: np.ndarray, title: str = "", vmin=None, vmax=None,
               cols: int = 5, cmap: str = "RdBu_r") -> None:
    if arr.ndim == 2:
        arr = arr[np.newaxis]
    T = arr.shape[0]
    rows = math.ceil(T / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 2.5 * rows))
    axes = np.array(axes).ravel()
    for t in range(T):
        axes[t].imshow(arr[t], origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
        axes[t].set_title(f"t={t}", fontsize=8)
        axes[t].axis("off")
    for ax in axes[T:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()


plot_field(batch.tgt[0].float().numpy(),   title="Ground truth x -- test batch")
plot_field(batch.input[0].float().numpy(), title="Observations y (NaN = unobserved)")


### Génération d'Ensemble — ODE vs SDE

**ODE :** toutes les étapes utilisent $\mathbf{x}_{t'} = c_\text{skip}\mathbf{x}_t + c_\text{out}\boldsymbol{\mu}_\phi$

**SDE :** à **chaque étape** $i$, on injecte un bruit calibré :

$$
\mathbf{x}_{t_{i+1}}^\text{SDE} = c_\text{skip}\,\mathbf{x}_{t_i} + c_\text{out}\,\boldsymbol{\mu}_\phi + \sigma_\phi(\mathbf{x}_{t_i},\,t_i)\,\boldsymbol{\varepsilon}_i, \quad \boldsymbol{\varepsilon}_i\sim\mathcal{N}(0,\mathbf{I})
$$

avec $\sigma_\phi(\mathbf{x},t) = \sigma_\text{noise}\cdot t^{\alpha(\mathbf{x})}$ — décroît de $\sigma_\text{noise}$ à $0$ au fil du débruitage.

La diversité ODE vient des différentes initialisations. La diversité SDE amplifie la diversité à chaque pas par la perturbation calibrée $\sigma_\phi(\mathbf{x},t)$.


In [ ]:
def sample_stoch_varcm(
    unet:        nn.Module,
    obs_cost:    nn.Module,
    prior_cost:  nn.Module,
    lambda_reg:  nn.Parameter,
    y:           Tensor,
    nsteps:      int   = 15,
    sigma_noise: float = 10.0,
    stoch:       bool  = False,       # False -> ODE | True -> SDE (noise at every step)
    schedule_power: float = 1.0,
    conditioning_mode: str = "obs",
    clip_denoised: bool = False,
) -> Tuple[Tensor, Tensor]:
    model_dtype = next(unet.parameters()).dtype
    y = y.to(dtype=model_dtype)
    B, C, H, W = y.shape
    device = y.device

    k = torch.linspace(0.0, 1.0, max(nsteps, 3), device=device)
    sigma_k = sigma_noise * (1.0 - k) ** schedule_power
    sigma_k = sigma_k.clamp(min=sigma_noise * 1e-4, max=sigma_noise).to(model_dtype)
    times = torch.cat([sigma_k / sigma_noise,
                       torch.zeros(1, device=device, dtype=model_dtype)])

    x = torch.randn((B, C, H, W), device=device, dtype=model_dtype) * sigma_noise
    trajectory = [x.float().cpu()]
    eps = 1e-3

    for i in range(len(times) - 1):
        t_cur  = torch.full((B,), times[i].item(),     dtype=model_dtype, device=device)
        t_prev = torch.full((B,), times[i + 1].item(), dtype=model_dtype, device=device)

        with torch.no_grad():
            x_next, log_sigma, _ = stoch_forward(
                unet, obs_cost, prior_cost, lambda_reg,
                x, y, t_cur, t_prev,
                sigma_noise=sigma_noise,
                conditioning_mode=conditioning_mode,
            )

        # SDE: inject calibrated noise at EVERY step
        if stoch:
            sigma_phi = torch.exp(log_sigma).view(B, 1, 1, 1).to(model_dtype)
            x_next = x_next + sigma_phi * torch.randn_like(x_next)

        if clip_denoised:
            x_next = x_next.clamp(-3.0, 3.0)

        x = x_next
        trajectory.append(x.float().cpu())

    return x.float().cpu(), torch.stack(trajectory, dim=0)


# ── Generate ensembles ────────────────────────────────────────────────────────
N_SAMPLES = 10
NSTEPS    = 40

y_dev = batch.input.to(device=device, dtype=dtype)

ens_ode, ens_sde = [], []
traj_ode, traj_sde = [], []

for mode, ens_list, traj_list in [("ODE", ens_ode, traj_ode), ("SDE", ens_sde, traj_sde)]:
    is_stoch = (mode == "SDE")
    seed_everything(42)
    for i in range(N_SAMPLES):
        smp, traj = sample_stoch_varcm(
            unet, obs_cost, prior_cost, lambda_reg, y_dev,
            nsteps=NSTEPS, sigma_noise=SIGMA_NOISE,
            stoch=is_stoch, schedule_power=SCHED_POW,
            conditioning_mode=COND_MODE_LOAD,
        )
        ens_list.append(smp)
        traj_list.append(traj)
        print(f"  {mode} sample {i+1}/{N_SAMPLES}", end="\r")
    print(f"\n[OK] {N_SAMPLES} {mode} members generated")

ens_ode_t = torch.stack(ens_ode, 0).squeeze(1)   # (N, C, H, W)
ens_sde_t = torch.stack(ens_sde, 0).squeeze(1)

mean_ode  = ens_ode_t.mean(0);  std_ode = ens_ode_t.std(0)
mean_sde  = ens_sde_t.mean(0);  std_sde = ens_sde_t.std(0)
gt_np     = batch.tgt[0].float().numpy()

print(f"\nODE ensemble std (mean): {std_ode.mean().item():.4f}")
print(f"SDE ensemble std (mean): {std_sde.mean().item():.4f}")

plot_field(gt_np,                              title="Ground truth x")
plot_field(mean_ode.numpy(),                   title="StochVarCM ODE -- ensemble mean")
plot_field(std_ode.numpy(),                    title="StochVarCM ODE -- ensemble std",  vmin=0, cmap="Reds")
plot_field(mean_sde.numpy(),                   title="StochVarCM SDE -- ensemble mean")
plot_field(std_sde.numpy(),                    title="StochVarCM SDE -- ensemble std",  vmin=0, cmap="Reds")
plot_field(np.abs(mean_ode.numpy() - gt_np),   title="|ODE mean - x|", vmin=0, cmap="Reds")
plot_field(np.abs(mean_sde.numpy() - gt_np),   title="|SDE mean - x|", vmin=0, cmap="Reds")


## Métriques — StochVarCM vs OI

- **mu-score** = $1 - \text{RMSE}/\sigma_{\text{GT}}$ (higher is better)
- **RMSE** (lower is better)
- **CRPS** ensemble (lower is better)
- **Spread-skill ratio** (ideal $\approx 1$)
- **ODE vs SDE** : comparaison des deux modes de sampling


In [ ]:
m_norm, s_norm = datamodule.norm_stats()

gt_np        = batch.tgt[0].float().numpy()
ens_ode_np   = ens_ode_t.numpy()   # (N, C, H, W)
ens_sde_np   = ens_sde_t.numpy()
mean_ode_np  = mean_ode.numpy()
mean_sde_np  = mean_sde.numpy()

# De-normalize
gt_phys      = gt_np       * s_norm + m_norm
mODE_phys    = mean_ode_np * s_norm + m_norm
mSDE_phys    = mean_sde_np * s_norm + m_norm
eODE_phys    = ens_ode_np  * s_norm + m_norm
eSDE_phys    = ens_sde_np  * s_norm + m_norm

start_t = datamodule.test_ds.indices[0]
ws      = datamodule.window_size
oi_phys = datamodule.oi[start_t : start_t + ws].astype(np.float32)
oi_norm = (oi_phys - m_norm) / s_norm


def compute_psd_radial(field: np.ndarray, dx: float = 1.0):
    f = field.copy(); f[np.isnan(f)] = 0.0
    psd_sum = sum(np.abs(np.fft.fft2(f[t])) ** 2 for t in range(f.shape[0]))
    psd_avg = psd_sum / f.shape[0]
    ny, nx  = psd_avg.shape
    yy, xx  = np.meshgrid(np.arange(ny)-ny//2, np.arange(nx)-nx//2, indexing="ij")
    rr      = np.sqrt(xx**2 + yy**2)
    r_max   = int(np.sqrt((ny//2)**2 + (nx//2)**2))
    radial  = np.array([
        np.mean(np.fft.fftshift(psd_avg)[(rr >= r) & (rr < r+1)])
        if np.any((rr >= r) & (rr < r+1)) else 0.0
        for r in range(r_max)
    ])
    threshold = 0.5 * np.max(radial[1:])
    idx_50    = np.where(radial < threshold)[0]
    lx = (nx * dx) / idx_50[0] if len(idx_50) > 0 else np.nan
    return lx, radial


def metrics_row(pred, gt, ens, label):
    gt_std  = np.nanstd(gt)
    rmse_v  = np.sqrt(np.nanmean((pred - gt)**2))
    mu_s    = 1.0 - rmse_v / gt_std
    spread  = np.nanmean(np.std(ens, axis=0))
    ssr     = spread / max(rmse_v, 1e-8)
    lx_gt,  _  = compute_psd_radial(gt)
    lx_pr, _   = compute_psd_radial(pred)
    row = dict(Method=label, mu_score=mu_s, RMSE=rmse_v, spread=spread,
               spread_skill=ssr, lambda_x_GT=lx_gt, lambda_x_pred=lx_pr)
    return row


rows = [
    metrics_row(mODE_phys, gt_phys, eODE_phys, "StochVarCM ODE (mean)"),
    metrics_row(mSDE_phys, gt_phys, eSDE_phys, "StochVarCM SDE (mean)"),
    metrics_row(oi_phys,   gt_phys, oi_phys[np.newaxis], "OI baseline"),
]

if HAS_PROPERSCORING:
    for row, ens_phys in zip(rows[:2], [eODE_phys, eSDE_phys]):
        crps_vals = []
        for t in range(gt_phys.shape[0]):
            for i in range(gt_phys.shape[1]):
                for j in range(gt_phys.shape[2]):
                    if not np.isnan(gt_phys[t, i, j]):
                        crps_vals.append(crps_ensemble(gt_phys[t,i,j], ens_phys[:,t,i,j]))
        row["CRPS"] = np.mean(crps_vals)

df = pd.DataFrame(rows).set_index("Method")
for col in ["mu_score", "RMSE", "spread", "spread_skill"]:
    df[col] = df[col].map(lambda v: f"{v:.4f}" if not (isinstance(v, float) and np.isnan(v)) else "N/A")
print("\n## Performance Assessment -- StochVarCM vs OI (SPDE dataset)\n")
print(df.to_markdown())
display(df)

# Spread-skill comparison table
print("\n=== Spread-skill summary ===")
for _, row_d in zip(range(2), rows[:2]):
    print(f"  {row_d['Method']:30s}  "
          f"RMSE={row_d['RMSE']:.4f}  spread={row_d['spread']:.4f}  "
          f"ratio={row_d['spread_skill']:.3f}")


### PSD Radiale — StochVarCM ODE / SDE / OI / GT


In [ ]:
_, psd_ode = compute_psd_radial(mODE_phys)
_, psd_sde = compute_psd_radial(mSDE_phys)
_, psd_oi  = compute_psd_radial(oi_phys)
_, psd_gt  = compute_psd_radial(gt_phys)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(psd_gt,  label="Ground truth x",         linewidth=2)
ax.semilogy(psd_ode, label="StochVarCM ODE (mean)",  linewidth=2, linestyle="--")
ax.semilogy(psd_sde, label="StochVarCM SDE (mean)",  linewidth=2, linestyle=":")
ax.semilogy(psd_oi,  label="OI baseline",            linewidth=2, linestyle="-.")
ax.set_xlabel("Radial wavenumber (pixels$^{-1}$)")
ax.set_ylabel("Power Spectral Density")
ax.set_title("Radial PSD -- SPDE dataset (StochVarCM)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### Comparaison Visuelle — GT / OI / ODE / SDE


In [ ]:
# Prior reconstruction
with torch.no_grad():
    _pc_dtype = next(prior_cost.parameters()).dtype
    tgt_recon = prior_cost.forward_ae(
        batch.tgt.to(device=device, dtype=_pc_dtype)
    ).float().cpu()[0]
tgt_recon_np = tgt_recon.numpy()

plot_field(gt_np,                                        title="Ground truth x")
plot_field(batch.input[0].float().numpy(),               title="Observations y (NaN -> 0)")
plot_field(oi_norm,                                      title="OI baseline (normalized)")
plot_field(tgt_recon_np,                                 title="Prior Phi(x) -- BilinAE")
plot_field(mean_ode_np,                                  title="StochVarCM ODE -- ensemble mean")
plot_field(std_ode.numpy(),                              title="StochVarCM ODE -- ensemble std", vmin=0, cmap="Reds")
plot_field(mean_sde_np,                                  title="StochVarCM SDE -- ensemble mean")
plot_field(std_sde.numpy(),                              title="StochVarCM SDE -- ensemble std", vmin=0, cmap="Reds")
plot_field(np.abs(mean_ode_np - gt_np),                  title="|ODE mean - x|", vmin=0, cmap="Reds")
plot_field(np.abs(mean_sde_np - gt_np),                  title="|SDE mean - x|", vmin=0, cmap="Reds")
plot_field(np.abs(oi_norm - gt_np),                      title="|OI - x|",       vmin=0, cmap="Reds")


### Processus de Transport — ODE vs SDE (membre 0)

Comparaison des trajectoires ODE et SDE pour visualiser :
- La convergence progressive du bruit vers la solution
- L'évolution de $\sigma_\phi(t;\mathbf{x}) = \sigma_\text{noise}\cdot t^{\alpha(\mathbf{x})}$ le long de la trajectoire


In [ ]:
import matplotlib.gridspec as gridspec

def plot_trajectory(traj_tensor, title="", channel=0, vmin=-2, vmax=2):
    proc_np = traj_tensor[:, 0, channel, :, :].numpy()
    nsteps_vis = proc_np.shape[0]
    cols = min(nsteps_vis, 6)
    rows = math.ceil(nsteps_vis / cols)
    fig = plt.figure(figsize=(3 * cols, 2.5 * rows + 1))
    gs = gridspec.GridSpec(rows + 1, cols,
                           height_ratios=[*([1] * rows), 0.08], hspace=0.2, wspace=0.05)
    axes = [fig.add_subplot(gs[i, j]) for i in range(rows) for j in range(cols)]
    cax  = fig.add_subplot(gs[rows, :])
    for k in range(nsteps_vis):
        im = axes[k].imshow(proc_np[k], origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
        axes[k].set_title(f"step {k}", fontsize=8)
        axes[k].axis("off")
    for ax in axes[nsteps_vis:]:
        ax.axis("off")
    plt.colorbar(im, cax=cax, orientation="horizontal").set_label("Field value")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()

plot_trajectory(traj_ode[0], title="ODE trajectory -- member 0, channel 0")
plot_trajectory(traj_sde[0], title="SDE trajectory -- member 0, channel 0")

# ── sigma(t) along the sampling trajectory ────────────────────────────────────

log_sigma_along = {"ODE": [], "SDE": []}

k_arr = torch.linspace(0.0, 1.0, max(NSTEPS, 3), device=device)
sigma_k = SIGMA_NOISE * (1.0 - k_arr) ** SCHED_POW
sigma_k = sigma_k.clamp(min=SIGMA_NOISE * 1e-4, max=SIGMA_NOISE).to(dtype)
times_traj = torch.cat([sigma_k / SIGMA_NOISE, torch.zeros(1, device=device, dtype=dtype)])

for mode, traj_list in [("ODE", traj_ode), ("SDE", traj_sde)]:
    with torch.no_grad():
        for step_idx in range(len(times_traj) - 1):
            x_step = traj_list[0][step_idx, :1].to(device=device, dtype=dtype)
            t_now  = times_traj[step_idx].view(1)
            t_next = times_traj[step_idx + 1].view(1)
            _, log_s, _ = stoch_forward(
                unet, obs_cost, prior_cost, lambda_reg,
                x_step, batch.input[:1].to(device=device, dtype=dtype),
                t_now, t_next,
                sigma_noise=SIGMA_NOISE, conditioning_mode=COND_MODE_LOAD,
            )
            log_sigma_along[mode].append(log_s.item())

steps_ax = np.arange(len(log_sigma_along["ODE"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for mode, ls_list, color in [("ODE", log_sigma_along["ODE"], "steelblue"),
                               ("SDE", log_sigma_along["SDE"], "darkorange")]:
    sig_arr = np.exp(ls_list)
    axes[0].plot(steps_ax, sig_arr, linewidth=2, color=color, label=mode)
    axes[1].plot(steps_ax, ls_list, linewidth=2, color=color, label=mode)

axes[0].set_xlabel("Sampling step"); axes[0].set_ylabel("$\\sigma_\\phi(t)$")
axes[0].set_title("Predicted $\\sigma_\\phi(t)$ along sampling trajectory")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Sampling step"); axes[1].set_ylabel("$\\log\\sigma_\\phi(t)$")
axes[1].set_title("$\\log\\sigma_\\phi(t)$ along sampling trajectory")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("sigma(t) decays as the state approaches the clean solution (step 0 = noise, last step = clean)", fontsize=10)
plt.tight_layout(); plt.show()

# ── Std collapse across ensemble members ────────────────────────────────────
step_stds_ode = np.array([
    traj_ode[i][k, 0].numpy().std(axis=(-2, -1)).mean()
    for i in range(N_SAMPLES)
    for k in [None]  # will redo below
])
# Proper computation:
step_stds_ode = np.array([
    [traj_ode[i][k, 0].numpy().std(axis=(-2, -1)).mean() for k in range(len(traj_ode[0]))]
    for i in range(N_SAMPLES)
])
step_stds_sde = np.array([
    [traj_sde[i][k, 0].numpy().std(axis=(-2, -1)).mean() for k in range(len(traj_sde[0]))]
    for i in range(N_SAMPLES)
])
steps = np.arange(step_stds_ode.shape[1])
gt_std = batch.tgt[0].float().numpy().std(axis=(-2, -1)).mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
for stds, color, label in [(step_stds_ode, "steelblue", "ODE"), (step_stds_sde, "darkorange", "SDE")]:
    mean_s = stds.mean(0);  q25 = np.percentile(stds, 25, 0);  q75 = np.percentile(stds, 75, 0)
    ax.fill_between(steps, q25, q75, alpha=0.2, color=color)
    ax.plot(steps, mean_s, linewidth=2, color=color, label=label)
ax.axhline(gt_std, color="green", linestyle="--", linewidth=1.5, label=f"GT spatial std={gt_std:.3f}")
ax.set_xlabel("Sampling step"); ax.set_ylabel("Mean spatial std")
ax.set_title("Variance collapse along sampling (step 0 = noise, last = clean)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
